# DATA-445: In-Class Assignment 6

Consider the `CarPrice_Assignment.csv` data file. This data is publicly available on the Kaggle website, and has information on cars (characteristics related to car dimensions, engine and more). The goal is to use car information to predict the price of the car. **In Python**, answer the following:

**1.** (5 points) Using the pandas library, read the csv data file and create a data-frame called `car_price`.

In [1]:
import pandas as pd

car_price = pd.read_csv('CarPrice_Assignment.csv')
car_price.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


**2.** (5 points) Define the `wheelbase`, `enginesize`, `compressionratio`, `horsepower`, `peakrpm`, `citympg`, and `highwaympg` as the predictor variables, and `price` is the target variable. Define the 5-fold cross-validation strategy.

In [9]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Lasso, LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

#defining the input and target variables

x = car_price[['wheelbase','enginesize','compressionratio', 'peakrpm', 'horsepower', 'citympg', 'highwaympg']]
y = car_price['price']

#defining the KFold cross-validation
skf = KFold(n_splits=5, shuffle=True, random_state=42)

**3.** (5 points) Consider the LASSO model model. Find the optimal value for $\lambda$ via cross-validation. Then, using the optimal value of $\lambda$ build a LASSO model for feature selection purposes.

In [7]:
# estimating the optimal value for lambda in Lasso

lasso_cv_md = make_pipeline(StandardScaler(),
                            LassoCV(cv=skf, n_jobs = -1))

lasso_cv_md.fit(x,y)

print(f"Optimal value for lambda: "
    f"{lasso_cv_md.named_steps['lassocv'].alpha_}")


Optimal value for lambda: 34.6717388981973


In [12]:
# Building the Lasso model with the optimal lambda

lasso_md = make_pipeline(StandardScaler(), 
                         Lasso(alpha=lasso_cv_md.named_steps['lassocv'].alpha_))
lasso_md.fit(x, y)

coef_df = pd.DataFrame({
    "Feature": x.columns,
    "Coefficient": lasso_md.named_steps['lasso'].coef_
})
print(coef_df)

            Feature  Coefficient
0         wheelbase  1028.227327
1        enginesize  4505.316336
2  compressionratio  1153.195649
3           peakrpm   926.302136
4        horsepower  1924.052371
5           citympg  -728.912606
6        highwaympg    -0.000000


**4.** (5 points) Using the train dataset, build a linear regression model with the selected features from part (3). Make sure to standardize the input features with `StandardScaler` and `Pipeline`. After that, use this model to perform a 5-fold cross-validation. Report the RMSE of this model.

In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = car_price[["wheelbase", "enginesize", "compressionratio", "horsepower", "peakrpm", "citympg"]].copy()
y = car_price["price"]

# Defining the linear model 
linear_md = make_pipeline(StandardScaler(), 
                          LinearRegression())

# Performing cross-validation to evaluate the linear model
cv_scores = cross_val_score(linear_md, X, y, cv=skf, scoring="neg_root_mean_squared_error", n_jobs=-1)
print(f"RMSE cross-validated score: {-cv_scores.mean()}")

RMSE cross-validated score: 3409.634150304407


**5.** (5 points) Using the train dataset, build a ridge regression model with the selected features from part (3). Make sure to standardize the input features with `StandardScaler` and `Pipeline`. After that, use this model to perform a 5-fold cross-validation. Report the RMSE of this model.

In [16]:
from sklearn.linear_model import RidgeCV

# Use the features selected by LASSO
X = x[["wheelbase", "enginesize", "compressionratio",
       "horsepower", "peakrpm", "citympg"]]

# Estimating lambda for ridge
ridge_cv = make_pipeline(StandardScaler(),
                         RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0], cv=skf))
ridge_cv.fit(X, y)
print(f"Best alpha for ridge: {ridge_cv.named_steps['ridgecv'].alpha_}")

Best alpha for ridge: 1.0


In [17]:
# Defining the Ridge model 
ridge_md = make_pipeline(StandardScaler(),
                         Ridge(alpha=1))

# Performing cross-validation to evaluate the Ridge model
cv_scores_ridge = cross_val_score(ridge_md, X, y, cv=skf, scoring="neg_root_mean_squared_error", n_jobs=-1)
print(f"RMSE cross-validated score for Ridge: {-cv_scores_ridge.mean()}")

RMSE cross-validated score for Ridge: 3407.3965520071956


**6.** (3 points) Using the results from parts (4) and (5), what model would you use to predict car prices? Explain.

Based on the results I would perfer to use the ridge model because it has the lower RMSE is smaller.